In [ ]:
pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 kB 6.9 MB/s eta 0:00:00


In [ ]:
# ============================================================
# CELL 1 — CHECK GPU + CHECKPOINT
# ============================================================

from google.colab import drive
import os
import torch

drive.mount("/content/drive")

LAST_MODEL = "/content/drive/MyDrive/AI_Training/Models/char_epoch2/best.pt"

print("=" * 60)
print("CHARACTER MODEL — RESUME TRAINING")
print("=" * 60)

print("\nCheckpoint:")
print(LAST_MODEL)

if not os.path.isfile(LAST_MODEL):
    raise FileNotFoundError(
        f"\nlast_1.pt not found:\n{LAST_MODEL}"
    )

print("✓ last_1.pt found")

print("\nGPU status:")
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )
else:
    raise RuntimeError(
        "\n❌ GPU is not available.\n"
        "Go to Runtime → Change runtime type → GPU."
    )

Mounted at /content/drive
CHARACTER MODEL — RESUME TRAINING

Checkpoint:
/content/drive/MyDrive/AI_Training/Models/char_epoch2/best.pt
✓ last_1.pt found

GPU status:
CUDA available: True
GPU: Tesla T4


In [ ]:
# ============================================================
# CELL 2 — EXTRACT AND VERIFY CLEANED DATASET
# ============================================================

from google.colab import drive
import os
import zipfile
import shutil

# ------------------------------------------------------------
# Mount Google Drive
# ------------------------------------------------------------

drive.mount("/content/drive")

# ------------------------------------------------------------
# EXACT ZIP PATH
# ------------------------------------------------------------

ZIP_PATH = (
    "/content/drive/MyDrive/"
    "AI_Training/Dataset/"
    "char_training_cleaned.zip"
)

# ------------------------------------------------------------
# LOCAL EXTRACTION PATH
# ------------------------------------------------------------

EXTRACT_PATH = "/content/char_training_dataset"

# ------------------------------------------------------------
# Check ZIP exists
# ------------------------------------------------------------

print("=" * 60)
print("DATASET CHECK")
print("=" * 60)

if not os.path.isfile(ZIP_PATH):
    raise FileNotFoundError(
        f"\n❌ ZIP file not found:\n{ZIP_PATH}"
    )

print("\n✓ ZIP found:")
print(ZIP_PATH)

# ------------------------------------------------------------
# Extract
# ------------------------------------------------------------

if os.path.exists(EXTRACT_PATH):

    print("\nExisting extracted dataset found.")
    print("Using existing extraction.")

else:

    print("\nExtracting dataset to Colab...")
    print("Please wait...\n")

    os.makedirs(
        EXTRACT_PATH,
        exist_ok=True
    )

    with zipfile.ZipFile(
        ZIP_PATH,
        "r"
    ) as zip_ref:

        zip_ref.extractall(
            EXTRACT_PATH
        )

    print("✓ Extraction completed.")

# ------------------------------------------------------------
# IMPORTANT:
# Find the actual folder containing images/ and labels/
# ------------------------------------------------------------

DATASET_ROOT = None

for root, dirs, files in os.walk(EXTRACT_PATH):

    if (
        "images" in dirs
        and "labels" in dirs
    ):

        possible_train = os.path.join(
            root,
            "images",
            "train"
        )

        possible_val = os.path.join(
            root,
            "images",
            "val"
        )

        if (
            os.path.isdir(possible_train)
            and os.path.isdir(possible_val)
        ):

            DATASET_ROOT = root
            break

if DATASET_ROOT is None:

    raise FileNotFoundError(
        "\n❌ Could not find the extracted "
        "dataset structure containing:\n"
        "images/train\n"
        "images/val\n"
        "labels/train\n"
        "labels/val"
    )

# ------------------------------------------------------------
# Set paths
# ------------------------------------------------------------

TRAIN_DIR = os.path.join(
    DATASET_ROOT,
    "images",
    "train"
)

VAL_DIR = os.path.join(
    DATASET_ROOT,
    "images",
    "val"
)

DATA_YAML = os.path.join(
    DATASET_ROOT,
    "data.yaml"
)

# ------------------------------------------------------------
# Verify
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("DATASET STRUCTURE")
print("=" * 60)

print("\nDataset root:")
print(DATASET_ROOT)

print("\nTrain folder:")
print(TRAIN_DIR)

print("\nValidation folder:")
print(VAL_DIR)

print("\ndata.yaml:")
print(DATA_YAML)

print("\nTrain folder exists:",
      os.path.isdir(TRAIN_DIR))

print("Validation folder exists:",
      os.path.isdir(VAL_DIR))

print("data.yaml exists:",
      os.path.isfile(DATA_YAML))

# ------------------------------------------------------------
# Count images
# ------------------------------------------------------------

IMAGE_EXTENSIONS = (
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
)

train_count = sum(
    1
    for f in os.listdir(TRAIN_DIR)
    if f.lower().endswith(
        IMAGE_EXTENSIONS
    )
)

val_count = sum(
    1
    for f in os.listdir(VAL_DIR)
    if f.lower().endswith(
        IMAGE_EXTENSIONS
    )
)

print("\n" + "=" * 60)
print("IMAGE COUNTS")
print("=" * 60)

print(
    f"\nTraining images:   {train_count:,}"
)

print(
    f"Validation images: {val_count:,}"
)

# ------------------------------------------------------------
# Final check
# ------------------------------------------------------------

if train_count == 141840 and val_count == 4013:

    print("\n✓ DATASET VERIFIED SUCCESSFULLY")

else:

    print("\n⚠️ Image counts differ from expected:")
    print("Expected train: 141,840")
    print("Expected val:     4,013")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DATASET CHECK

✓ ZIP found:
/content/drive/MyDrive/AI_Training/Dataset/char_training_cleaned.zip

Extracting dataset to Colab...
Please wait...

✓ Extraction completed.

DATASET STRUCTURE

Dataset root:
/content/char_training_dataset/clean

Train folder:
/content/char_training_dataset/clean/images/train

Validation folder:
/content/char_training_dataset/clean/images/val

data.yaml:
/content/char_training_dataset/clean/data.yaml

Train folder exists: True
Validation folder exists: True
data.yaml exists: True

IMAGE COUNTS

Training images:   141,840
Validation images: 4,013

✓ DATASET VERIFIED SUCCESSFULLY


In [ ]:
# ============================================================
# CELL 3 — CHECK GPU AND PREVIOUS CHECKPOINT
# ============================================================

import os
import torch
from google.colab import drive

drive.mount("/content/drive")

LAST_MODEL = (
    "/content/drive/MyDrive/"
    "AI_Training/Models/last_1.pt"
)

print("=" * 60)
print("RESUME TRAINING CHECK")
print("=" * 60)

print("\nCheckpoint:")
print(LAST_MODEL)

if not os.path.isfile(LAST_MODEL):
    raise FileNotFoundError(
        f"\n❌ last_1.pt not found:\n{LAST_MODEL}"
    )

print("✓ last_1.pt found")

print("\nGPU:")
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():

    print(
        "GPU name:",
        torch.cuda.get_device_name(0)
    )

else:

    raise RuntimeError(
        "\n❌ GPU is not available.\n"
        "Go to Runtime → Change runtime type → GPU."
    )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
RESUME TRAINING CHECK

Checkpoint:
/content/drive/MyDrive/AI_Training/Models/last_1.pt
✓ last_1.pt found

GPU:
CUDA available: True
GPU name: Tesla T4


In [ ]:
# ============================================================
# CELL 4 — TRAIN ONE ADDITIONAL EPOCH
# START FROM PREVIOUSLY LEARNED BEST WEIGHTS
# ============================================================

from ultralytics import YOLO
import torch
import os

# ============================================================
# PATHS
# ============================================================

MODEL_PATH = (
    "/content/drive/MyDrive/"
    "AI_Training/Models/best_1.pt"
)

DATA_YAML = (
    "/content/char_training_dataset/"
    "clean/data.yaml"
)

# ============================================================
# CHECK MODEL
# ============================================================

print("=" * 60)
print("CHARACTER MODEL — ADDITIONAL TRAINING")
print("=" * 60)

if not os.path.isfile(MODEL_PATH):
    raise FileNotFoundError(
        f"\n❌ best_1.pt not found:\n{MODEL_PATH}"
    )

if not os.path.isfile(DATA_YAML):
    raise FileNotFoundError(
        f"\n❌ data.yaml not found:\n{DATA_YAML}"
    )

print("\n✓ Starting weights:")
print(MODEL_PATH)

print("\n✓ Dataset:")
print(DATA_YAML)

# ============================================================
# GPU
# ============================================================

if not torch.cuda.is_available():
    raise RuntimeError(
        "\n❌ CUDA GPU is not available."
    )

print("\n✓ GPU:")
print(torch.cuda.get_device_name(0))

# ============================================================
# LOAD PREVIOUSLY TRAINED WEIGHTS
# ============================================================

model = YOLO(MODEL_PATH)

print("\n✓ Previous learned weights loaded.")

# ============================================================
# TRAIN ONE NEW EPOCH
# ============================================================

print("\n" + "=" * 60)
print("STARTING ONE ADDITIONAL EPOCH")
print("=" * 60)

results = model.train(

    data=DATA_YAML,

    # Exactly ONE new epoch
    epochs=1,

    imgsz=640,

    batch=-1,

    device=0,

    workers=4,

    amp=True,

    cache=False,

    save=True,

    save_period=1,

    project="/content/char_training",

    name="char_yolo26s_epoch2",

    exist_ok=True,

    # IMPORTANT:
    # This is NOT resume=True.
    # We are fine-tuning from best_1.pt.
    resume=False,

    verbose=True
)

print("\n" + "=" * 60)
print("ADDITIONAL EPOCH FINISHED")
print("=" * 60)

CHARACTER MODEL — ADDITIONAL TRAINING

✓ Starting weights:
/content/drive/MyDrive/AI_Training/Models/best_1.pt

✓ Dataset:
/content/char_training_dataset/clean/data.yaml

✓ GPU:
Tesla T4

✓ Previous learned weights loaded.

STARTING ONE ADDITIONAL EPOCH
Ultralytics 8.4.152 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/char_training_dataset/clean/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.01

In [ ]:
# ============================================================
# ZIP LATEST TRAINING MODELS → GOOGLE DRIVE
# ============================================================

import os
import shutil

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

TRAINING_DIR = (
    "/content/char_training/"
    "char_yolo26s_epoch2"
)

WEIGHTS_DIR = os.path.join(
    TRAINING_DIR,
    "weights"
)

DRIVE_DIR = (
    "/content/drive/MyDrive/"
    "AI_Training/Models"
)

ZIP_NAME = "char_yolo26s_epoch2_models"

# ------------------------------------------------------------
# CHECK GOOGLE DRIVE
# ------------------------------------------------------------

if not os.path.exists("/content/drive/MyDrive"):
    raise RuntimeError(
        "❌ Google Drive is not mounted."
    )

# ------------------------------------------------------------
# CHECK TRAINING DIRECTORY
# ------------------------------------------------------------

if not os.path.isdir(TRAINING_DIR):
    raise FileNotFoundError(
        f"❌ Training directory not found:\n{TRAINING_DIR}"
    )

if not os.path.isdir(WEIGHTS_DIR):
    raise FileNotFoundError(
        f"❌ Weights directory not found:\n{WEIGHTS_DIR}"
    )

# ------------------------------------------------------------
# CREATE DRIVE DIRECTORY
# ------------------------------------------------------------

os.makedirs(
    DRIVE_DIR,
    exist_ok=True
)

# ------------------------------------------------------------
# SHOW MODELS
# ------------------------------------------------------------

print("=" * 70)
print("LATEST TRAINING MODELS")
print("=" * 70)

model_files = []

for filename in sorted(os.listdir(WEIGHTS_DIR)):

    filepath = os.path.join(
        WEIGHTS_DIR,
        filename
    )

    if os.path.isfile(filepath):

        size_mb = (
            os.path.getsize(filepath)
            / (1024 * 1024)
        )

        print(
            f"✓ {filename:<25} "
            f"{size_mb:.2f} MB"
        )

        model_files.append(filename)

if not model_files:
    raise RuntimeError(
        "❌ No model files found."
    )

# ------------------------------------------------------------
# CREATE ZIP
# ------------------------------------------------------------

zip_base = os.path.join(
    DRIVE_DIR,
    ZIP_NAME
)

print("\n" + "=" * 70)
print("CREATING ZIP")
print("=" * 70)

created_zip = shutil.make_archive(
    zip_base,
    "zip",
    WEIGHTS_DIR
)

# ------------------------------------------------------------
# VERIFY ZIP
# ------------------------------------------------------------

if not os.path.isfile(created_zip):
    raise RuntimeError(
        "❌ ZIP creation failed."
    )

zip_size_mb = (
    os.path.getsize(created_zip)
    / (1024 * 1024)
)

print("\n" + "=" * 70)
print("✓ ZIP CREATED SUCCESSFULLY")
print("=" * 70)

print("\nZIP file:")
print(created_zip)

print(
    f"\nZIP size: {zip_size_mb:.2f} MB"
)

print("\nModels included:")

for filename in model_files:
    print(f"  ✓ {filename}")

print("\nGoogle Drive location:")
print(
    "/content/drive/MyDrive/"
    "AI_Training/Models/"
    f"{ZIP_NAME}.zip"
)

print("\n" + "=" * 70)
print("BACKUP COMPLETE")
print("=" * 70)

LATEST TRAINING MODELS
✓ best.pt                   19.42 MB
✓ epoch0.pt                 76.84 MB
✓ last.pt                   19.42 MB

CREATING ZIP

✓ ZIP CREATED SUCCESSFULLY

ZIP file:
/content/drive/MyDrive/AI_Training/Models/char_yolo26s_epoch2_models.zip

ZIP size: 105.00 MB

Models included:
  ✓ best.pt
  ✓ epoch0.pt
  ✓ last.pt

Google Drive location:
/content/drive/MyDrive/AI_Training/Models/char_yolo26s_epoch2_models.zip

BACKUP COMPLETE
